<a href="https://colab.research.google.com/github/Venni2911/LogicMojo-AI-ML-Sept25-VenniRaj/blob/main/Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Attention**

# **Agenda**

* **Introduction to Attention**
* **Basic Concepts of Attention**
* **Types of Attention Mechanisms**
* **Self-Attention Mechanism**
* **Multi-Head Attention**
* **Attention in Transformers**
* **Attention Variants**
* **Applications of Attention Mechanisms**

# **Introduction to Attention**

The Attention mechanism is a crucial concept in deep learning, particularly in the context of natural language processing (NLP), sequence-to-sequence models and computer vision. It allows models to focus on specific parts of the input sequence when producing each part of the output sequence, thereby improving performance in tasks like translation, summarization, Introduced to address the limitations of traditional sequence models like recurrent neural networks (RNNs) and convolutional neural networks (CNNs), attention mechanisms enable models to focus on the most relevant parts of the input when generating output, improving performance and interpretability.



### **The Need for Attention**

**Traditional models like RNNs and CNNs face several challenges:**
- **Long-Range Dependencies:** RNNs struggle to capture dependencies in long sequences due to issues like vanishing gradients.
- **Fixed Context Window:** CNNs and RNNs typically use a fixed-size context window, limiting their ability to consider varying parts of the input for different outputs.
- **Sequential Processing:** RNNs process sequences sequentially, leading to inefficiencies in training and inference.
- Attention mechanisms address these challenges by allowing models to dynamically focus on different parts of the input sequence, providing a flexible and efficient way to handle complex data.



# **Basic Concepts of Attention**
- **Context Vector:** Represents a weighted sum of the input features, where the weights reflect the importance of each input feature in producing the current output.
- **Alignment Scores:** Determine the relevance of each input element with respect to the current output element. These scores are computed using various functions, such as dot-product or additive (Bahdanau) attention.
- **Attention Weights:** Normalized alignment scores (usually using a softmax function) that indicate the importance of each input element. The sum of attention weights is 1.

### **Context Vector:**
- The context vector is a fixed-size representation that summarizes relevant information from the entire input sequence.
- In attention mechanisms, the context vector is typically a weighted sum of the input sequence elements, where the weights are derived from the attention scores.

**How it is Used:**
- The context vector is used in various sequence-to-sequence models to provide a dynamic context to the decoder at each step of the generation process.
- By focusing on different parts of the input sequence for each output element, the context vector helps in producing more accurate and contextually relevant outputs.

**Consider a sequence-to-sequence model for translation. The context vector helps the decoder focus on the relevant parts of the input sentence while generating each word in the output sentence.**

In [ ]:
import torch

# Example of context vector calculation in a simplified attention mechanism
def calculate_context_vector(attention_weights, encoder_outputs):
    # attention_weights: [batch_size, sequence_length]
    # encoder_outputs: [batch_size, sequence_length, hidden_size]

    # Calculate context vector as the weighted sum of encoder outputs
    context_vector = torch.sum(attention_weights.unsqueeze(-1) * encoder_outputs, dim=1)

    return context_vector

# Dummy data for illustration
attention_weights = torch.tensor([[0.1, 0.2, 0.3, 0.4]])
encoder_outputs = torch.tensor([[[0.1, 0.2], [0.2, 0.3], [0.3, 0.4], [0.4, 0.5]]])

context_vector = calculate_context_vector(attention_weights, encoder_outputs)
print(context_vector)


tensor([[0.3000, 0.4000]])


### **Alignment Scores:**

**Alignment Scores: Different Methods to Calculate Alignment Scores**

- **Alignment Scores:**
Alignment scores measure the relevance or importance of each input element with respect to the current output element being generated.
- These scores are used to compute the attention weights, which determine how much focus the model should place on each input element.

**Methods to Calculate Alignment Scores:**

**Dot-Product Attention:**
- Computes the alignment score as the dot product of the query and key vectors.
- Efficient but may have large variance for large-dimensional vectors.

In [ ]:
import torch

def dot_product_attention(query, key):
    # query: [batch_size, hidden_size]
    # key: [batch_size, sequence_length, hidden_size]
    return torch.matmul(query.unsqueeze(1), key.transpose(-2, -1)).squeeze(1)

query = torch.tensor([[0.1, 0.2]])
key = torch.tensor([[[0.1, 0.2], [0.2, 0.3], [0.3, 0.4], [0.4, 0.5]]])
alignment_scores = dot_product_attention(query, key)
print(alignment_scores)


tensor([[0.0500, 0.0800, 0.1100, 0.1400]])


**Scaled Dot-Product Attention:**
- Similar to dot-product attention but scales the scores by the square root of the dimensionality to counteract the large variance.

In [ ]:
import torch
import math

def scaled_dot_product_attention(query, key, d_k):
    # query: [batch_size, hidden_size]
    # key: [batch_size, sequence_length, hidden_size]
    scores = torch.matmul(query.unsqueeze(1), key.transpose(-2, -1)).squeeze(1)
    return scores / math.sqrt(d_k)

query = torch.tensor([[0.1, 0.2]])
key = torch.tensor([[[0.1, 0.2], [0.2, 0.3], [0.3, 0.4], [0.4, 0.5]]])
d_k = query.size(-1)
alignment_scores = scaled_dot_product_attention(query, key, d_k)
print(alignment_scores)


tensor([[0.0354, 0.0566, 0.0778, 0.0990]])


###**Attention Weights:**



- Attention weights are derived by applying a softmax function to the alignment scores, converting them into a probability distribution.
- These weights determine the importance of each input element for generating the current output element.

**How to Use:**
- Use the attention weights to compute a weighted sum of the input elements, resulting in the context vector.

In [ ]:
import torch
import torch.nn.functional as F

# Example of deriving and using attention weights
def calculate_attention_weights(alignment_scores):
    return F.softmax(alignment_scores, dim=-1)

# Dummy alignment scores for illustration
alignment_scores = torch.tensor([[0.1, 0.2, 0.3, 0.4]])

# Derive attention weights
attention_weights = calculate_attention_weights(alignment_scores)
print(attention_weights)

# Use attention weights to compute context vector
def calculate_context_vector(attention_weights, encoder_outputs):
    return torch.sum(attention_weights.unsqueeze(-1) * encoder_outputs, dim=1)

encoder_outputs = torch.tensor([[[0.1, 0.2], [0.2, 0.3], [0.3, 0.4], [0.4, 0.5]]])
context_vector = calculate_context_vector(attention_weights, encoder_outputs)
print(context_vector)


tensor([[0.2138, 0.2363, 0.2612, 0.2887]])
tensor([[0.2625, 0.3625]])


# **Types of Attention Mechanisms**

###**Additive (Bahdanau) Attention**

**Mechanism:**
- Additive attention, introduced by Bahdanau et al., uses a feedforward neural network to calculate alignment scores.
- It concatenates the decoder's previous hidden state with each encoder output and passes this through a neural network to compute the alignment scores.
- The scores are then passed through a softmax function to obtain the attention weights.

**Steps:**
- Concatenate the decoder hidden state with each encoder hidden state.
- Pass the concatenated vectors through a feedforward neural network to get the alignment scores.
- Apply softmax to the alignment scores to get the attention weights.
- Compute the context vector as a weighted sum of the encoder hidden states.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size):
        super(AdditiveAttention, self).__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Parameter(torch.rand(hidden_size))

    def forward(self, query, key):
        # query: [batch_size, hidden_size]
        # key: [batch_size, sequence_length, hidden_size]
        query = query.unsqueeze(1).expand_as(key)
        energy = torch.tanh(self.attn(torch.cat([query, key], dim=-1)))
        energy = torch.matmul(energy, self.v.unsqueeze(-1)).squeeze(-1)
        return energy

# Example usage
query = torch.tensor([[0.1, 0.2]])
key = torch.tensor([[[0.1, 0.2], [0.2, 0.3], [0.3, 0.4], [0.4, 0.5]]])
attention = AdditiveAttention(hidden_size=2)
alignment_scores = attention(query, key)
attention_weights = F.softmax(alignment_scores, dim=-1)
print("Alignment Scores:", alignment_scores)
print("Attention Weights:", attention_weights)


Alignment Scores: tensor([[0.0156, 0.0185, 0.0215, 0.0244]], grad_fn=<SqueezeBackward1>)
Attention Weights: tensor([[0.2489, 0.2496, 0.2504, 0.2511]], grad_fn=<SoftmaxBackward0>)


### **Dot-Product (Luong) Attention**

**Mechanism:**
- Dot-product attention, introduced by Luong et al., computes the alignment scores as the dot product of the decoder hidden state and the encoder hidden states.
- It is computationally efficient as it avoids the need for an additional neural network to compute the scores.

**Steps:**
- Compute the dot product between the decoder hidden state and each encoder hidden state.
- Apply softmax to the alignment scores to get the attention weights.
- Compute the context vector as a weighted sum of the encoder hidden states.

In [ ]:
import torch

def dot_product_attention(query, key):
    # query: [batch_size, hidden_size]
    # key: [batch_size, sequence_length, hidden_size]
    return torch.matmul(query.unsqueeze(1), key.transpose(-2, -1)).squeeze(1)

# Example usage
query = torch.tensor([[0.1, 0.2]])
key = torch.tensor([[[0.1, 0.2], [0.2, 0.3], [0.3, 0.4], [0.4, 0.5]]])
alignment_scores = dot_product_attention(query, key)
attention_weights = F.softmax(alignment_scores, dim=-1)
print("Alignment Scores:", alignment_scores)
print("Attention Weights:", attention_weights)


Alignment Scores: tensor([[0.0500, 0.0800, 0.1100, 0.1400]])
Attention Weights: tensor([[0.2389, 0.2461, 0.2536, 0.2614]])


### **Scaled Dot-Product Attention**

**Mechanism:**
- Scaled dot-product attention is similar to dot-product attention but introduces a scaling factor to prevent the dot products from growing too large, which can cause gradients to vanish or explode.
- The scaling factor is the square root of the dimensionality of the key vectors, which stabilizes the gradients and improves training.

**Steps:**
- Compute the dot product between the decoder hidden state and each encoder hidden state.
- Scale the dot products by the square root of the dimensionality of the key vectors.
- Apply softmax to the alignment scores to get the attention weights.
- Compute the context vector as a weighted sum of the encoder hidden states.

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(query, key, d_k):
    # query: [batch_size, hidden_size]
    # key: [batch_size, sequence_length, hidden_size]
    scores = torch.matmul(query.unsqueeze(1), key.transpose(-2, -1)).squeeze(1)
    return scores / math.sqrt(d_k)

# Example usage
query = torch.tensor([[0.1, 0.2]])
key = torch.tensor([[[0.1, 0.2], [0.2, 0.3], [0.3, 0.4], [0.4, 0.5]]])
d_k = query.size(-1)
alignment_scores = scaled_dot_product_attention(query, key, d_k)
attention_weights = F.softmax(alignment_scores, dim=-1)
print("Alignment Scores:", alignment_scores)
print("Attention Weights:", attention_weights)


Alignment Scores: tensor([[0.0354, 0.0566, 0.0778, 0.0990]])
Attention Weights: tensor([[0.2421, 0.2473, 0.2526, 0.2580]])


# **Self-Attention Mechanism**

### **Self-Attention Definition:**
- Self-attention, also known as intra-attention, is a mechanism that allows each position in the input sequence to attend to all other positions. It computes the relevance of each word in a sequence to every other word, enabling the model to capture long-range dependencies and contextual relationships more effectively.

### **Importance in Transformer Models:**
- **Contextual Understanding:** Self-attention helps in understanding the context of a word based on other words in the sequence, which is crucial for tasks like language modeling, translation, and more.
- **Parallelization:** Unlike RNNs, which process sequences sequentially, transformers with self-attention can process all words in a sequence simultaneously, making training much faster.
- **Capturing Long-Range Dependencies:** Self-attention can capture relationships between distant words in a sequence, which is challenging for traditional RNNs and CNNs.


### Implementation of the self-attention mechanism in PyTorch:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert (
            self.head_dim * heads == embed_size
        ), "Embedding size needs to be divisible by heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        # Split the embedding into self.heads different pieces
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        # Scaled dot-product attention
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.embed_size ** (1 / 2)), dim=3)

        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        out = self.fc_out(out)
        return out

# Example usage
embed_size = 256
heads = 8
values = torch.randn((64, 50, embed_size))
keys = torch.randn((64, 50, embed_size))
queries = torch.randn((64, 50, embed_size))
mask = None

self_attention = SelfAttention(embed_size, heads)
out = self_attention(values, keys, queries, mask)
print(out.shape)  # Output: torch.Size([64, 50, 256])


torch.Size([64, 50, 256])


- This implementation demonstrates the core steps of the self-attention mechanism, including the creation of the Q, K, and V matrices, the scaled dot-product calculation, and the aggregation of context vectors. It can be extended and modified to fit more complex models and tasks.

# **Multi-Head Attention**

**Multi-Head Attention**

**Concept**
**Why Multi-Head Attention is Used:**
- Multi-head attention is a crucial component of the Transformer model architecture. It involves running multiple self-attention mechanisms, or "heads," in parallel. Each head operates on different parts of the input data, capturing various aspects of relationships and dependencies within the data.

**Improvements in Model Performance:**
- **Rich Representations:** Different heads can focus on different parts of the sentence, learning various relationships and dependencies.
- **Capture Multiple Features:** Allows the model to capture multiple features from different subspaces at different positions.
- **Avoiding Information Bottleneck:** By using multiple heads, the model can capture more fine-grained details, reducing the risk of losing important information.


###**Architecture**
**Linear Projections:**
- The input embeddings are linearly projected into queries (Q), keys (K), and values (V) for multiple heads.
- These projections are done using different learned weight matrices for each head.


**Scaled Dot-Product Attention:**

**Each head performs scaled dot-product attention separately:**
- Compute the dot products of the query with all keys to get the raw attention scores.
- Apply a softmax function to obtain the attention weights.
- Multiply the weights by the value vectors to get the output of the attention head.

**Concatenation and Final Linear Layer:**
- The outputs of the individual attention heads are concatenated.
- This concatenated output is then projected through a final linear layer to produce the final output.


**Architecture Diagram:**
- Split the input into multiple heads.
- Perform attention for each head independently.
- Concatenate the outputs from all heads.
- Pass the concatenated outputs through a linear layer.

### **Implementation of multi-head attention in PyTorch:**

In [ ]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert (
            self.head_dim * heads == embed_size
        ), "Embedding size needs to be divisible by heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        # Split the embedding into self.heads different pieces
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        # Scaled dot-product attention for each head
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        out = self.fc_out(out)
        return out

# Example usage
embed_size = 256
heads = 8
values = torch.randn((64, 50, embed_size))
keys = torch.randn((64, 50, embed_size))
queries = torch.randn((64, 50, embed_size))
mask = None

multi_head_attention = MultiHeadAttention(embed_size, heads)
out = multi_head_attention(values, keys, queries, mask)
print(out.shape)  # Output: torch.Size([64, 50, 256])


torch.Size([64, 50, 256])


This implementation captures the essence of multi-head attention, including the splitting of the input embeddings into multiple heads, performing scaled dot-product attention for each head, and finally concatenating and transforming the outputs to produce the final result.

#**Attention in Transformers**

Transformers, introduced in the paper "Attention is All You Need" by Vaswani et al., revolutionized the field of natural language processing by utilizing attention mechanisms extensively. Here's a detailed look at how attention mechanisms are integrated into transformers, the importance of positional encoding, and code snippets to illustrate these concepts.




### **Transformer Architecture**

**Overview:**
- Transformers are built on the idea of self-attention mechanisms, which allow each word in a sequence to pay attention to other words in the sequence. The architecture consists of an encoder and a decoder, both of which are composed of multiple layers of self-attention and feedforward neural networks.

- **Encoder:** Processes the input sequence and generates a representation for each position.
- **Decoder:** Generates the output sequence based on the encoder's representation and previously generated outputs.


**The key components of a transformer include:**

- **Self-Attention Mechanism:** Calculates attention scores for each word in relation to every other word in the sequence.
- **Multi-Head Attention:** Uses multiple self-attention mechanisms in parallel to capture different aspects of the input.
- **Positional Encoding:** Adds information about the position of each word in the sequence.
- **Feedforward Neural Networks:** Applies non-linear transformations to the attention outputs.
- **Layer Normalization and Residual Connections:** Stabilize training and improve gradient flow.


### **Positional Encoding**
Transformers do not have an inherent sense of the order of words because they do not process the input sequentially. To incorporate the order of words, positional encoding is added to the input embeddings.

**Importance:**
Provides information about the position of words in the sequence.
- Helps the model distinguish between different permutations of the same set of words.



###**Implementation: Positional encodings**

In [ ]:
import torch
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_size, 2).float() * (-math.log(10000.0) / embed_size))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1), :]

# Example usage
embed_size = 512
max_len = 100
positional_encoding = PositionalEncoding(embed_size, max_len)
input_embeddings = torch.randn(64, 100, embed_size)  # (batch_size, sequence_length, embed_size)
encoded_input = positional_encoding(input_embeddings)
print(encoded_input.shape)  # Output: torch.Size([64, 100, 512])


torch.Size([64, 100, 512])


**Transformer Model Using Attention**


In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_size, 2).float() * (-math.log(10000.0) / embed_size))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1), :]

# Example usage
embed_size = 512
max_len = 100
positional_encoding = PositionalEncoding(embed_size, max_len)
input_embeddings = torch.randn(64, 100, embed_size)  # (batch_size, sequence_length, embed_size)
encoded_input = positional_encoding(input_embeddings)
print(encoded_input.shape)  # Output: torch.Size([64, 100, 512])

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert (
            self.head_dim * heads == embed_size
        ), "Embedding size needs to be divisible by heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        out = self.fc_out(out)
        return out

class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        attention = self.attention(value, key, query, mask)

        x = self.dropout(self.norm1(attention + query))
        forward = self.feed_forward(x)
        out = self.dropout(self.norm2(forward + x))
        return out

class Transformer(nn.Module):
    def __init__(
        self,
        embed_size,
        num_layers,
        heads,
        device,
        forward_expansion,
        dropout,
        max_length,
    ):
        super(Transformer, self).__init__()
        self.embed_size = embed_size
        self.device = device
        self.word_embedding = nn.Embedding(10000, embed_size)
        self.position_embedding = PositionalEncoding(embed_size, max_length)

        self.layers = nn.ModuleList(
            [
                TransformerBlock(
                    embed_size,
                    heads,
                    dropout=dropout,
                    forward_expansion=forward_expansion,
                )
                for _ in range(num_layers)
            ]
        )

        self.fc_out = nn.Linear(embed_size, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        N, seq_length = x.shape
        positions = torch.arange(0, seq_length).expand(N, seq_length).to(self.device)
        out = self.dropout(self.word_embedding(x) + self.position_embedding(self.word_embedding(positions)))

        for layer in self.layers:
            out = layer(out, out, out, mask)

        out = self.fc_out(out.mean(dim=1))
        return out

# Example usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_size = 512
num_layers = 6
heads = 8
dropout = 0.1
forward_expansion = 4
max_length = 100

model = Transformer(embed_size, num_layers, heads, device, forward_expansion, dropout, max_length).to(device)
input_data = torch.randint(0, 10000, (64, 100)).to(device)  # Example input
mask = None  # Replace with an actual mask if needed
output = model(input_data, mask)
print(output.shape)  # Output: torch.Size([64, 1])


torch.Size([64, 100, 512])
torch.Size([64, 1])


# **Attention Variants**

### **Global vs. Local Attention:**

- **Global Attention:** In global attention, all elements of the input sequence contribute to the output at each decoding step. It computes attention scores between the decoder hidden state and all encoder hidden states.

- **Local Attention:** In local attention, only a subset of the input sequence contributes to the output at each decoding step. This subset is determined dynamically based on the current position of the decoder. It improves efficiency and is useful when processing long sequences.



### **Soft vs. Hard Attention:**

- **Soft Attention:** Soft attention computes a weighted sum of all input elements, where the weights are determined by attention scores. It is differentiable and allows for end-to-end training via backpropagation.

- **Hard Attention:** Hard attention selects only one or a few input elements to attend to, based on certain criteria. It is non-differentiable and requires reinforcement learning or other techniques for training. Hard attention can be more computationally efficient but is less flexible than soft attention.

### **Self-Attention vs. Cross-Attention:**

- **Self-Attention:** Self-attention computes attention scores within a single sequence, allowing each element to attend to other elements within the same sequence. It is widely used in transformer architectures for capturing dependencies between different positions in the input sequence.

- **Cross-Attention:** Cross-attention computes attention scores between elements of two different sequences. It is commonly used in tasks like machine translation, where the model needs to attend to both the source and target sequences simultaneously. Cross-attention allows the model to align relevant information between the input and output sequences.



# **Applications of Attention Mechanisms**

### **Machine Translation:**
- Attention mechanisms improve translation models by allowing them to focus on relevant parts of the source sentence when generating each word of the target sentence. Traditional sequence-to-sequence models without attention often struggle with long sentences because they have to compress all the information into fixed-length representations. With attention, the model can selectively attend to different parts of the source sentence, alleviating this bottleneck. This enables better translation quality, especially for long or complex sentences, as the model can capture dependencies between distant words more effectively.



### **Text Summarization:**

- Attention mechanisms play a crucial role in summarizing long texts by helping the model identify the most important information. When summarizing a lengthy document, it's essential to focus on key phrases and sentences to capture the essence of the text concisely. Attention mechanisms allow the model to assign higher weights to relevant parts of the input document, ensuring that the generated summary retains the most salient information while omitting less critical details. This results in more informative and coherent summaries.

###**Image Captioning:**
- In image captioning tasks, attention mechanisms enable the model to focus on different regions of the input image when generating captions. Instead of treating the entire image equally, attention mechanisms allow the model to dynamically attend to relevant visual features while generating each word of the caption. This enables the model to generate more accurate and detailed captions that describe the content of the image effectively. By attending to specific regions of interest, the model can produce captions that are more descriptive and contextually relevant.

###**Question Answering:**
- Attention mechanisms are instrumental in question answering (QA) systems by helping the model locate relevant information in the input passage when generating answers to questions. When presented with a question, the QA model needs to identify the most relevant parts of the passage that contain the answer. Attention mechanisms allow the model to assign higher weights to words or phrases in the passage that are semantically related to the question, enabling it to extract the necessary information more effectively. This enhances the accuracy and relevance of the answers generated by the QA system.